## **PPG訊號量測實作單元(二)：簡單照一下就知道壓力大不大啦！我是說……血壓！**

> 在這次的實驗中，鐵克會帶大家學習實務上如何對訊號進行微分

> 並搭配ECG波峰偵測得到兩點的時間差來取得Pulse Arrival Time (PAT)！

> 最後試著用統計上的最小平方法估算自己的收縮血壓值是多少吧！

# 0. 先收錄一些會用到的公式和函式吧！

In [1]:
#@title
from google.colab import files
import plotly.graph_objects as go
import numpy as np
import pandas as pd
from scipy.signal import find_peaks
import math
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
def find_peak_weird_detection(data, height, distance):
  peak_list = find_peaks(data, height=height, distance=distance)[0]
  if(peak_list.size != 0):
    print('總共找到', peak_list.size, '個Peak')
    peak_peak_list = (np.diff(peak_list))
    pp_mean = np.mean(peak_peak_list)
    counter = 0
    counter2 = 0
    counter3 = 0

    for i in peak_peak_list:
      if(i > pp_mean+100):
        counter2 = counter2 + 1
    
    print('發現有',counter2,'個「怪怪Peak」')

    weird_peak_list_start = np.zeros(counter2,int)
    weird_peak_list_end = np.zeros(counter2,int)
    weird_pp_list = np.zeros(counter2,int)
  

    for i in peak_peak_list:
      if(i > pp_mean+100):
        weird_peak_list_start[counter3] = peak_list[counter]
        weird_peak_list_end[counter3] = peak_list[counter+1]
        weird_pp_list[counter3] = i
        counter3 += 1
      counter = counter + 1

    pp_mean_fixed_1 = (pp_mean*(peak_peak_list.size))
    pp_mean_fixed_2 = pp_mean_fixed_1 - np.sum(weird_pp_list)
    pp_mean_fixed = int(pp_mean_fixed_2/(peak_peak_list.size - weird_pp_list.size))

  else:
    print('沒有找到任何Peak，請重新調整height和distance來尋找peak。')

  return peak_list, peak_peak_list, weird_peak_list_start, weird_peak_list_end, pp_mean_fixed 

def cut_first_data(data):
  data = data[1:]
  return data

# 1.將範例ECG訊號和PPG訊號丟進程式庫吧！

直接一次點選兩個檔案即可唷

In [ ]:
uploaded = files.upload()

for fn in uploaded.keys():
  print('你已經上傳了','"{name}" '.format(
      name=fn, length=len(uploaded[fn])))

# 2.看看剛剛丟進來的ECG和PPG訊號吧！

In [4]:
#來讀取剛上傳的資料吧！
df = pd.read_csv('範例心電訊號.txt') #這邊要記得改成自己檔案的檔名唷！
a1 = np.array(df)
a2 = a1[0 : , 0]

#來讀取剛上傳的資料吧！
df2 = pd.read_csv('範例PPG訊號.txt') #這邊要記得改成自己檔案的檔名唷！
b1 = np.array(df2)
b2 = b1[0 : , 0]

In [ ]:
#稍微設定一下畫布資訊！
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=a2,
    name='Original Plot'
))
fig.update_layout(
    title="同步測得的ECG訊號",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
#秀出心電訊號吧！
fig.show() 

In [ ]:
#稍微設定一下畫布資訊！
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=b2,
    name='Original Plot'
))
fig.update_layout(
    title="同步測得的PPG訊號",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
#秀出PPG訊號吧！
fig.show() 

# 3. 透過波峰偵測功能先把ECG訊號的波峰抓出來吧！

> 執行後若有發現怪怪Peak，試著調整height和distance吧！

> 觀察上面的ECG訊號，height調整可以參考波峰高度平均落在Y軸的多少呢？

> distance調整可以參考波峰之間間距從X軸上來看大概是多少呢？

In [ ]:
height = 63
distance = 600
(peak_list_a2, pp_list_a2, weird_p_start_a2, weird_p_end_a2, pp_mean_a2) = find_peak_weird_detection(a2, height, distance)

In [ ]:
#再次檢驗看看檢測到的波形吧！
length = 40000
#稍微設定一下畫布資訊！
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=a2[0:length],
    mode='lines',
    name='心電訊號'
))

counter = 0
for i in peak_list_a2:
  counter += 1
  if(i >= length):
    break

#把偵測到的波峰也標記在畫布上吧！
fig.add_trace(go.Scatter(
    x=peak_list_a2[0:counter],
    y=[a2[j] for j in peak_list_a2],
    mode='markers',
    marker=dict(
        size=8,
        color='red',
        symbol='cross'
    ),
    name='分析之Peak位置'
))
fig.update_layout(
    title="同步測得的ECG訊號與波峰偵測結果",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
#秀出你的成果吧！
fig.show()

# 4. 採用So and Chan的演算法計算出PPG訊號的各點的斜率
# ，進一步使用波峰偵測把斜率波峰位置抓出來！

> 斜率的主要概念是用來了解變化程度的快慢，一般常見的兩點式(該點減前一點)。

> So and Chan 的演算法則是強化了這個概念，延展兩點式，將該點的前後兩點也納進考量！

> 其公式為 Slope(n) = -2*X(n-2) - 1*X(n-1) + 1*X(n+1) + 2*X(n+2)

In [9]:
counter = 2 #從資料原點開始沒有X(n-2)的資料，因此要從第二筆資料開始計算

b3 = np.zeros(np.size(b2)-2) #建一個空的資料空間來放置PPG訊號的斜率
while(counter < np.size(b2)-2):
  b3[counter] = (-2)*b2[counter-2] + (-1)*b2[counter-1] + 1*b2[counter+1] + 2*b2[counter+2]
  counter += 1

In [ ]:
#再次檢驗看看檢測到的波形吧！
length2 = 40000
#稍微設定一下畫布資訊！
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=b3[0:length],
    mode='lines',
    name='Original Plot'
))
fig.update_layout(
    title="以So and chan計算出的PPG訊號斜率",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
#秀出你的成果吧！
fig.show()

> 執行後若有發現怪怪Peak，試著調整height和distance吧！

> 觀察上面的PPG訊號的斜率，height調整可以參考波峰高度平均落在Y軸的多少呢？

> distance調整可以參考波峰之間間距從X軸上來看大概是多少呢？

In [ ]:
height2 = 4
distance2 = 600
(peak_list_b3, pp_list_b3, weird_p_start_b3, weird_p_end_b3, pp_mean_b3) = find_peak_weird_detection(b3, height2, distance2)

In [ ]:
#再次檢驗看看檢測到的波形吧！
length2 = 40000
#稍微設定一下畫布資訊！
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=b3[0:length2],
    mode='lines',
    name='PPG訊號的So and chan斜率'
))

counter2 = 0
for i in peak_list_b3:
  counter2 += 1
  if(i >= length2):
    break

#把偵測到的波峰也標記在畫布上吧！
fig.add_trace(go.Scatter(
    x=peak_list_b3[0:counter2],
    y=[b3[j] for j in peak_list_b3],
    mode='markers',
    marker=dict(
        size=8,
        color='red',
        symbol='cross'
    ),
    name='分析之Peak位置'
))

fig.update_layout(
    title="同步測得的PPG訊號斜率與波峰偵測結果",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
#秀出你的成果吧！
fig.show()

# 5. 確認ECG和PPG的波峰資料

> 為何需要確認波峰資料呢？並不是因為擔心程式有問題，而是擔心資料並不完整！

> 想像如果你當時按下「紀錄」的瞬間，正巧ECG訊號波峰已經過去，但PPG訊號斜率波峰剛好到來瞬間！又或是那個瞬間是你按下「停止紀錄」的瞬間。

> 這兩種情況都會使資料多一筆或少一筆，甚至無法對齊，當然這也能夠透過更複雜的演算法克服，不過這邊就讓初學的我們用肉眼觀察吧！

In [ ]:
#先用肉眼觀察ECG訊號和PPG訊號斜率的詳細資料

print('ECG訊號的波峰位置是在',peak_list_a2)
print('PPG訊號斜率的波峰位置是在',peak_list_b3)
print('ECG訊號的波峰個數是',np.size(peak_list_a2))
print('PPG訊號斜率的波峰個數是',np.size(peak_list_b3))

#注意！！！！！！！下段程式非必要不能執行！！！！！

此段程式碼將會剔除PPG訊號的第一筆資料

In [ ]:
#注意！！！！！！！此段程式非必要不能執行！！！！！
#第一筆PPG資料比第一筆ECG資料大才需要執行
#觀察範例訊號是第一筆PPG訊號是245，第一筆ECG訊號是480
#判斷為合理的資料的話無須執行此段程式
peak_list_b3 = cut_first_data(peak_list_b3) #將PPG訊號的第一筆資料刪除

In [ ]:
#再次確認ECG訊號和PPG訊號斜率的詳細資料

print('ECG訊號的波峰位置是在',peak_list_a2)
print('PPG訊號斜率的波峰位置是在',peak_list_b3)
print('ECG訊號的波峰個數是',np.size(peak_list_a2))
print('PPG訊號斜率的波峰個數是',np.size(peak_list_b3))

#6. 正式分析Pulse Arrival Time (PAT) 資料

In [ ]:
#事先訂好Pulse Arrival Time的個數，避免兩筆資料的個數不同
#因為是要將兩筆資料相減，若有人比較多，則取較小個數的那個人

PAT_list_number = 0
if(np.size(peak_list_b3) >= np.size(peak_list_a2)):
  PAT_list_number = np.size(peak_list_a2)
else:
  PAT_list_number = np.size(peak_list_b3)

print('設定好Pulse Arrival Time的個數為', PAT_list_number)

In [ ]:
PAT_list = np.zeros(PAT_list_number) #依前面設定將PAT資料的空間事先建立好
counter = 0
while counter < PAT_list_number:
    print('第',counter+1,'筆的波峰相減得出的PAT為',peak_list_b3[counter] - peak_list_a2[counter],'毫秒')
    PAT_list[counter] = peak_list_b3[counter] - peak_list_a2[counter] 
    counter = counter + 1

PAT_mean = np.mean(PAT_list)
print("\n") #輸出換行指令
print("所有PAT資料的平均為", round(PAT_mean,2),'毫秒')

# 6. 依據Pulse Arrival Time以先前模擬好的算式來估算血壓值

> 模擬算式並非難事，只需要紀錄PAT時搭配血壓計量測，將兩方資料進行迴歸計算即可

> 由於同學們可能身邊沒有血壓計，因此鐵克這邊先以自己的資料舉例說明

> 血壓估算一元一次方程式：BP = A * (d^2 / PAT^2) + B

In [19]:
#以下是鐵克事先用血壓計建立的十筆資料，量測血壓110mm/Hg時的PAT為241.38毫秒，依此類推
#同學若想建立自己的模型，可先量血壓後立刻量測並記錄ECG/PPG
#反覆進行數次得到幾組數據後，透過前面段落的程式算出個別血壓時的PAT，再直接修改這邊的對應資料即可

BP = np.array([110, 127, 115, 120, 131, 125, 117, 121, 120, 126])

PAT = np.array([241.38, 228.85, 233.43, 223.22, 214.45, 225.2, 218.68, 219.19, 255, 236.93]) #單位：毫秒

In [ ]:
#建立一元一次方程式 BP = A * (d^2 / PAT^2) + B 所需變數

secPAT = PAT*0.001                 #將PAT的單位調整回為"秒"
length = 1.75                    #單位：公尺 這邊同學可以改成自己的身高！
d2PAT2 = pow(length*0.6 ,2) / pow(secPAT, 2) #將身高乘上換算手長的比例0.6，與PAT進行運算

print(d2PAT2)

In [ ]:
#進行迴歸計算前，先來用圖片感受資料的相關性吧！

def create_screen():
  # create a figure and axes
  fig = plt.figure(figsize=(12,5))
  ax1 = plt.subplot(1,2,1)   

  # set up the subplots as needed
  ax1.set_xlim(( 0, 40))            
  ax1.set_ylim((80, 200))
  ax1.set_xlabel('d^2 / PAT^2')
  ax1.set_ylabel('Blood Pressure')

  txt_title = ax1.set_title('')

  BP_plot, = ax1.plot(d2PAT2, BP,   'o', color = 'blue') 

  ax1.legend(['BP_plot']);
  return fig,ax1,txt_title,BP_plot
create_screen()

>從上圖中可以觀察到，血壓值與資料呈現正相關

In [ ]:
#將BP和(d^2 / PAT^2)進行迴歸計算！
lm = LinearRegression()
lm.fit(np.reshape(d2PAT2, (len(d2PAT2), 1)), np.reshape(BP, (len(BP), 1)))

# 印出係數
print(lm.coef_)

# 印出截距
print(lm.intercept_)

 
print('迴歸計算出一元一次方程式為 BP=',lm.coef_[0,0],'* d2PAT2 +',lm.intercept_[0] )

In [23]:
#接下來只需要將此次量測PAT平均值代入即可利用方程式回推自己的血壓估計值囉！

print('此次量測算出的平均PAT為',round(PAT_mean,2) ,'毫秒')
secPAT_mean = PAT_mean * 0.001 #修改單位為'秒'
d2PAT2_this_time = pow(length*0.6 ,2) / pow(secPAT_mean, 2)

BP_estimate = (lm.coef_[0,0] * d2PAT2_this_time) + lm.intercept_[0]

print('此次估算的血壓為',round(BP_estimate,2) ,'mm/Hg')

此次量測算出的平均PAT為 221.18 毫秒
此次估算的血壓為 122.9 mm/Hg


# 章節總結：
---
1.   血液抵達手指的瞬間需要先微分PPG訊號再取波峰位置。
2.   血壓和PAT確實有一定的相關性。
3.   要血壓方程式最準確，每個人都需要事先用血壓計的資料微調屬於自己的方程式係數與截距。


# 恭喜完成本章節課程



> 有需要的同學可以使用密碼搭配書籍網站取得課程證書

> 證書密碼：「yutechEOGPPG3」！